#### Extraction des caractéristiques avec model_vgg16_fullyConv_sans_augm_couche_nongelee_batch_16.pt | menage EHCVM 2018 extraction des caractéristiques de leurs images en 2021

Ce script réalise l’extraction de caractéristiques compactes (4096 dimensions) à partir d’images satellites à l’aide d’un modèle Fully Convolutional basé sur VGG16. Voici les étapes principales expliquées :

Chargement des données et préparation des chemins : Le script commence par définir les chemins des fichiers nécessaires, notamment le modèle sauvegardé (model_path), le répertoire contenant les images de test (test_image_dir), le fichier CSV contenant les métadonnées associées aux images (csv_path), et l'emplacement où sauvegarder le fichier final avec les caractéristiques extraites (output_path).

Définition du modèle Fully Convolutional : Une classe VGGFullyConv est définie pour adapter l'architecture VGG16. Les couches convolutives (self.features) extraient des caractéristiques riches, et les couches de classification convolutive (self.classifier) compressent ces caractéristiques à 4096 dimensions via deux convolutions 1x1. Une réduction spatiale moyenne (mean(dim=(2, 3))) est ensuite appliquée pour condenser les informations spatiales.

Chargement du modèle et mise en mode évaluation : Le modèle est instancié et ses poids sont chargés depuis un fichier pré-entraîné. Il est ensuite transféré sur un GPU ou CPU selon la disponibilité, et configuré pour le mode évaluation (désactivation du calcul des gradients).

Transformation des images : Un pipeline de transformation est défini pour normaliser les images en entrée afin qu'elles soient compatibles avec VGG16. Les images sont redimensionnées à 224x224 pixels, converties en tenseurs, et normalisées en utilisant les moyennes et écarts-types des canaux RGB du jeu ImageNet.

Vérification des dimensions des caractéristiques : Une image factice (aléatoire) est passée dans le modèle pour vérifier que la sortie après réduction est de taille [1, 4096]. Cette étape permet de s'assurer que le modèle fonctionne comme prévu.

Extraction des caractéristiques compactes : Pour chaque image du fichier CSV, le script :

Vérifie si le fichier image existe dans le répertoire.
Charge et transforme l’image avec le pipeline défini.
Passe l’image transformée dans le modèle Fully Convolutional pour extraire un vecteur de caractéristiques compactes de taille 4096.
Si une image est manquante ou corrompue, un vecteur de zéros est utilisé par défaut.
Fusion des caractéristiques avec le fichier CSV : Les vecteurs de caractéristiques extraits pour chaque image sont combinés avec les métadonnées existantes du fichier CSV. Chaque vecteur (4096 dimensions) est ajouté comme nouvelles colonnes, nommées feature_0 à feature_4095.

Sauvegarde des résultats : Le DataFrame final, contenant les métadonnées et les caractéristiques extraites, est sauvegardé dans un fichier CSV à l’emplacement spécifié (output_path). Le script confirme la sauvegarde par un message.

En résumé, ce script transforme des images satellites en vecteurs de caractéristiques compactes de 4096 dimensions, facilitant ainsi leur utilisation pour des tâches d’analyse ou de modélisation ultérieures. Les étapes de gestion des erreurs et de fusion garantissent la robustesse et l’intégrité des résultats.

In [ ]:
import os
import pandas as pd
import torch
from torchvision import transforms
from PIL import Image
import numpy as np
from torch import nn
from torchvision import models
from tqdm import tqdm
import warnings
import gc
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings('ignore')


# Chemins des fichiers et répertoires
model_path = r"D:\wealth_predict_sentinel\models\model_vgg16_fullyConv_sans_augm_couche_gelee_batch_16.pt" 
test_image_dir = r"D:\wealth_predict_sentinel\Data\downloaded\Image_satellite_base_EHCVM_2018_Zoom_14_Sentinel_2_pour_an_2021" # menage EHCVM 2018 leurs images en 2021
csv_path = r"D:\wealth_predict_sentinel\Data\processed_csv\Data_EHCVM_2018_with_images_names.csv"
output_path = r"D:\wealth_predict_sentinel\Data\processed_csv\images_with_features_extracted_4096_fullyConv_sans_augm_couche_gelee_batch_16_base_EHCVM_2018_pour_an_2021_version_2.csv" #On met à jour ce nom quand on met de nouvelles données

# Configuration du device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Utilisation du device: {device}")

# Définition de l'architecture du modèle - identique à l'entraînement
class VGGFullyConv(nn.Module):
    def __init__(self, num_classes=4):
        super(VGGFullyConv, self).__init__()
        self.features = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1).features
        
        # Geler les couches convolutives comme dans l'entraînement
        for param in self.features.parameters():
            param.requires_grad = False
            
        self.classifier = nn.Sequential(
            nn.Conv2d(512, 4096, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(4096, 4096, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(4096, num_classes, kernel_size=1)
        )

    def forward(self, x):
        # Extraction jusqu'à l'avant-dernière couche ReLU
        x = self.features(x)
        x = self.classifier[0](x)  # Première Conv2d
        x = self.classifier[1](x)  # Premier ReLU
        x = self.classifier[2](x)  # Deuxième Conv2d
        x = self.classifier[3](x)  # Deuxième ReLU
        return x.mean(dim=(2, 3))  # Moyenne spatiale pour obtenir le vecteur de 4096 caractéristiques

# Transformation des images - identique à l'entraînement
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def load_image(image_path):
    """Charge et transforme une image."""
    try:
        with Image.open(image_path) as img:
            img = img.convert("RGB")
            return transform(img)
    except Exception as e:
        print(f"Erreur lors du chargement de l'image {image_path}: {e}")
        return None

def extract_image_features(model, image_tensor):
    """Extrait les caractéristiques d'une image."""
    image_tensor = image_tensor.unsqueeze(0).to(device)
    with torch.no_grad():
        features = model(image_tensor)
    return features.cpu().numpy().flatten()

def main():
    try:
        # 1. Chargement et configuration du modèle
        print("Chargement du modèle...")
        model = VGGFullyConv(num_classes=4).to(device)
        model.load_state_dict(torch.load(model_path))
        model.eval()

        # 2. Vérification des dimensions
        print("Vérification des dimensions des caractéristiques...")
        dummy_image = torch.randn(1, 3, 224, 224).to(device)
        with torch.no_grad():
            dummy_features = model(dummy_image)
        print(f"Dimensions des caractéristiques: {dummy_features.shape}")
        del dummy_image, dummy_features
        torch.cuda.empty_cache()

        # 3. Chargement des données
        print("Chargement des données...")
        df = pd.read_csv(csv_path)
        total_images = len(df)
        print(f"Nombre total d'images à traiter: {total_images}")

        # 4. Extraction des caractéristiques
        print("Début de l'extraction des caractéristiques...")
        features_list = []
        for idx, row in tqdm(df.iterrows(), total=total_images):
            image_name = row['nom de l\'image']
            image_path = os.path.join(test_image_dir, image_name)

            if os.path.exists(image_path):
                image_tensor = load_image(image_path)
                if image_tensor is not None:
                    features = extract_image_features(model, image_tensor)
                else:
                    features = np.zeros(4096)
            else:
                print(f"Image introuvable: {image_name}")
                features = np.zeros(4096)

            features_list.append(features)
            
            # Nettoyage périodique de la mémoire
            if idx % 100 == 0:
                torch.cuda.empty_cache()
                gc.collect()

        # 5. Création et sauvegarde du DataFrame
        print("Création du DataFrame final...")
        features_df = pd.DataFrame(
            features_list,
            columns=[f"feature_{i}" for i in range(4096)]
        )
        
        df_with_features = pd.concat(
            [df.reset_index(drop=True), features_df.reset_index(drop=True)],
            axis=1
        )

        print("Sauvegarde des résultats...")
        df_with_features.to_csv(output_path, index=False)
        print(f"Extraction terminée. Résultats sauvegardés dans: {output_path}")

    except Exception as e:
        print(f"Une erreur est survenue: {str(e)}")
        import traceback
        traceback.print_exc()
    finally:
        torch.cuda.empty_cache()
        gc.collect()

if __name__ == "__main__":
    main()

In [ ]:
pd.read_csv(r"D:\wealth_predict_sentinel\Data\processed_csv\images_with_features_extracted_4096_fullyConv_sans_augm_couche_gelee_batch_16_base_EHCVM_2018_pour_an_2021_version_2.csv")
